# Natural Language Processing - Assignment 2

### Group 35

Members:


*   Mehdi Hellal
*   Antonio Augusto Brito de Sousa
*   Adam Oprchal


## Introduction

In this assignment, we use Hugging Face transformers for the same classification task as in the first assignment. We find pre-trained models that are suitable and promising for fine-tuning (ideally pre-trained for the same language and/or text genre).

This is the description of the dataset taken from the first assignment:

[The Biggest Spam Ham Phish Email Dataset (250000+)](https://www.kaggle.com/datasets/akshatsharma2/the-biggest-spam-ham-phish-email-dataset-300000). This dataset contains over 250000 English messages and emails annotated into three classes: spam (unwanted mass message/email), ham (safe, normal message/email) and phish (malicious type of spam designed to steal sensitive information).

This dataset was created by merging other publicly available and open-source datasets and properly cleaned and deduplicated. Labeling of the data was also verified. The labels in the dataset have this mapping: 0 → Ham, 1 → Phish, 2 → Spam.

## Exploratory Data Analysis and Preprocessing

Since we performed an exploratory data analysis in the first assignment, we will not be repeating the same steps. We will however drop duplicates from the dataset and reduce its size just like before. The size reduction is performed with the goal of balancing classes in the dataset and to also make this assignment comparable to the first one.

Any further preprocessing (like lemmatization and stemming) is not used, because transformers are quite good at working with natural text.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("akshatsharma2/the-biggest-spam-ham-phish-email-dataset-300000")

In [ ]:
import pandas as pd

dataset = pd.read_csv(path + "/df.csv")
dataset.head()

In [ ]:
dataset.shape

In [ ]:
dataset = dataset.dropna().drop_duplicates()
dataset.shape

In [ ]:
mapping = {0: "ham", 1: "phish", 2: "spam"}

dataset["named_label"] = dataset["label"].map(mapping)

In [ ]:
dataset["named_label"].value_counts().plot(kind="bar")

In [ ]:
min_count = dataset["label"].value_counts().min()
n_samples = int(min_count * 0.1)

# Sample the same number of messages from each class to keep the task balanced
# and small enough for a short Colab T4 run.
dataset = (
    dataset.groupby("label", group_keys=False)
      .sample(n=n_samples, random_state=42)
      .reset_index(drop=True)
)

dataset["named_label"].value_counts()

In [ ]:
dataset["named_label"].value_counts().plot(kind="bar", title="Balanced Class Distribution")

Now we turn the balanced pandas dataframe into Hugging Face datasets. We use a fixed seed and stratified splitting so that the train, validation and test sets keep the same class proportions.

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

SEED = 42

train_df, temp_df = train_test_split(
    dataset,
    test_size=0.10,
    random_state=SEED,
    stratify=dataset["label"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

train_valid_test_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True), preserve_index=False),
    "validation": Dataset.from_pandas(valid_df.reset_index(drop=True), preserve_index=False),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True), preserve_index=False),
})

train_valid_test_dataset

In [ ]:
for split_name, split_dataset in train_valid_test_dataset.items():
    print(f"{split_name} label counts:")
    print(pd.Series(split_dataset["label"]).map(mapping).value_counts().sort_index())
    print()

In [ ]:
train_valid_test_dataset["train"][0]

## Model Selection and Training

The task is English email/message classification, so BERT-family encoder models are a sensible fit: they were pre-trained on large English corpora and can be fine-tuned for sequence classification. We avoid models already fine-tuned for spam or phishing detection because that would make the comparison with Assignment 1 less fair.

We compare three full fine-tuning runs:

* `distilbert/distilbert-base-uncased`: a smaller distilled BERT model and our fast transformer baseline.
* `google-bert/bert-base-uncased`: the standard uncased BERT baseline.
* `FacebookAI/roberta-base`: a RoBERTa encoder trained with an optimized masked language modeling procedure.

As a bonus, we also run parameter-efficient fine-tuning with LoRA on BERT and compare its performance and trainable parameter count with full fine-tuning.

In [ ]:
!pip -q install evaluate accelerate "peft==0.14.0"

In [ ]:
import gc
import random
import re

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CLASS_NAMES = ["Ham", "Phish", "Spam"]
ID2LABEL = {0: "ham", 1: "phish", 2: "spam"}
LABEL2ID = {label: idx for idx, label in ID2LABEL.items()}

accuracy_metric = evaluate.load("accuracy")

In [ ]:
def slugify(text):
    return re.sub(r"[^a-z0-9]+", "-", text.lower()).strip("-")


def count_parameters(model):
    total_params = sum(parameter.numel() for parameter in model.parameters())
    trainable_params = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    return trainable_params, total_params


def compute_classification_metrics(labels, predictions):
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_macro": precision_score(labels, predictions, average="macro", zero_division=0),
        "recall_macro": recall_score(labels, predictions, average="macro", zero_division=0),
        "f1_macro": f1_score(labels, predictions, average="macro", zero_division=0),
    }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    metrics = compute_classification_metrics(labels, predictions)
    # Keep the Trainer's default accuracy key simple while also reporting macro metrics.
    metrics["accuracy"] = accuracy_metric.compute(
        predictions=predictions,
        references=labels,
    )["accuracy"]
    return metrics


def build_trainer(model, tokenizer, tokenized_dataset, training_args):
    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": tokenized_dataset["train"],
        "eval_dataset": tokenized_dataset["validation"],
        "data_collator": DataCollatorWithPadding(tokenizer=tokenizer),
        "compute_metrics": compute_metrics,
    }

    try:
        return Trainer(processing_class=tokenizer, **trainer_kwargs)
    except TypeError:
        # Older transformers versions used tokenizer= instead of processing_class=.
        return Trainer(tokenizer=tokenizer, **trainer_kwargs)

In [ ]:
def run_experiment(config, dataset_dict):
    display_name = config["display_name"]
    model_name = config["model_name"]
    max_length = config.get("max_length", 128)

    print(f"\n===== {display_name} =====")
    print(f"Loading tokenizer and model from {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    def preprocess_function(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_length)

    tokenized_dataset = dataset_dict.map(preprocess_function, batched=True)
    removable_columns = [
        column
        for column in ["text", "named_label"]
        if column in tokenized_dataset["train"].column_names
    ]
    tokenized_dataset = tokenized_dataset.remove_columns(removable_columns)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    if config.get("use_lora", False):
        print("Applying LoRA adapters")
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            inference_mode=False,
            r=8,
            lora_alpha=16,
            lora_dropout=0.1,
            target_modules=["query", "value"],
            modules_to_save=["classifier"],
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()

    trainable_params, total_params = count_parameters(model)

    training_args = TrainingArguments(
        output_dir=f"./results/{slugify(display_name)}",
        learning_rate=config.get("learning_rate", 2e-5),
        per_device_train_batch_size=config.get("batch_size", 16),
        per_device_eval_batch_size=config.get("batch_size", 16),
        num_train_epochs=config.get("epochs", 3),
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        logging_strategy="epoch",
    )

    trainer = build_trainer(model, tokenizer, tokenized_dataset, training_args)

    print("Training")
    train_output = trainer.train()

    print("Evaluating on the test set")
    test_output = trainer.predict(tokenized_dataset["test"])
    predictions = np.argmax(test_output.predictions, axis=-1)
    labels = np.array(test_output.label_ids)
    metrics = compute_classification_metrics(labels, predictions)
    matrix = confusion_matrix(labels, predictions, labels=[0, 1, 2])

    result = {
        "model": display_name,
        "model_id": model_name,
        "uses_lora": config.get("use_lora", False),
        "accuracy": metrics["accuracy"],
        "precision_macro": metrics["precision_macro"],
        "recall_macro": metrics["recall_macro"],
        "f1_macro": metrics["f1_macro"],
        "train_runtime": train_output.metrics.get("train_runtime", np.nan),
        "trainable_params": trainable_params,
        "total_params": total_params,
        "predictions": predictions,
        "labels": labels,
        "confusion_matrix": matrix,
    }

    print(
        f"Accuracy: {result['accuracy']:.4f} | "
        f"Macro F1: {result['f1_macro']:.4f} | "
        f"Trainable params: {trainable_params:,}/{total_params:,}"
    )

    del trainer, model, tokenized_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

In [ ]:
experiment_configs = [
    {
        "display_name": "DistilBERT",
        "model_name": "distilbert/distilbert-base-uncased",
        "max_length": 128,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 2e-5,
    },
    {
        "display_name": "BERT",
        "model_name": "google-bert/bert-base-uncased",
        "max_length": 128,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 2e-5,
    },
    {
        "display_name": "RoBERTa",
        "model_name": "FacebookAI/roberta-base",
        "max_length": 128,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 2e-5,
    },
    {
        "display_name": "BERT + LoRA",
        "model_name": "google-bert/bert-base-uncased",
        "use_lora": True,
        "max_length": 128,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 2e-5,
    },
]

In [ ]:
experiment_results = []

for experiment_config in experiment_configs:
    experiment_results.append(
        run_experiment(experiment_config, train_valid_test_dataset)
    )

The full fine-tuning experiments completed successfully for DistilBERT, BERT and RoBERTa. The LoRA experiment did not complete because of a Colab dependency conflict between `peft` and the preinstalled `torchao` package.

During model loading, Hugging Face reported some `MISSING` and `UNEXPECTED` keys. This is expected in our setup because the original checkpoints were pre-trained for language modeling, while we are loading them for sequence classification. The original language-modeling heads are discarded, and a new classification head is initialized for the three spam/ham/phish labels.

## Results

The table below collects the test-set metrics for every transformer experiment. Precision, recall and F1 are macro-averaged so that each class contributes equally.

In [ ]:
results_df = pd.DataFrame([
    {
        key: value
        for key, value in result.items()
        if key not in {"predictions", "labels", "confusion_matrix"}
    }
    for result in experiment_results
])

results_display = results_df.copy()
for column in ["accuracy", "precision_macro", "recall_macro", "f1_macro"]:
    results_display[column] = (results_display[column] * 100).round(2)

results_display["train_runtime"] = results_display["train_runtime"].round(2)
results_display["trainable_params"] = results_display["trainable_params"].map(lambda value: f"{value:,}")
results_display["total_params"] = results_display["total_params"].map(lambda value: f"{value:,}")

results_display.rename(columns={
    "model": "Model",
    "model_id": "Hugging Face model",
    "uses_lora": "LoRA",
    "accuracy": "Accuracy (%)",
    "precision_macro": "Macro Precision (%)",
    "recall_macro": "Macro Recall (%)",
    "f1_macro": "Macro F1 (%)",
    "train_runtime": "Train Runtime (s)",
    "trainable_params": "Trainable Params",
    "total_params": "Total Params",
})

## Confusion Matrix Heatmaps

The heatmaps show raw test-set counts. Rows are true labels and columns are predicted labels.

In [ ]:
def plot_confusion_heatmap(result, ax=None, vmax=None, show_colorbar=True):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4))

    sns.heatmap(
        result["confusion_matrix"],
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        vmin=0,
        vmax=vmax,
        cbar=show_colorbar,
        ax=ax,
    )
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(f"{result['model']} Confusion Matrix")
    return ax


for result in experiment_results:
    plot_confusion_heatmap(result)
    plt.tight_layout()
    plt.show()

The confusion matrices show strong diagonal patterns for all three completed transformer models, meaning most test examples are classified correctly. DistilBERT has the most off-diagonal errors, while BERT and RoBERTa reduce these mistakes noticeably. RoBERTa performs best overall, with the fewest expected misclassifications.

Because accuracy and macro F1 are very close for each model, performance is balanced across the three classes rather than being driven by one easy class. The remaining errors are likely concentrated between spam and phishing, since these categories share similar language and intent.

## Comparison With Assignment 1

The first assignment used sparse Bag-of-Words features and dense Word2Vec representations. The best recorded Assignment 1 result was the three-layer Word2Vec + MLP model at 91.44% accuracy.

In [ ]:
assignment1_results = pd.DataFrame({
    "Model": [
        "BoW + Naive Bayes",
        "BoW + Logistic Regression",
        "Tuned BoW + Logistic Regression",
        "Word2Vec + Logistic Regression",
        "Word2Vec + Random Forest",
        "Word2Vec + MLP, 2 layers",
        "Word2Vec + MLP, 3 layers",
        "Tuned Word2Vec + MLP",
    ],
    "Approach": [
        "Assignment 1 - Sparse",
        "Assignment 1 - Sparse",
        "Assignment 1 - Sparse",
        "Assignment 1 - Dense",
        "Assignment 1 - Dense",
        "Assignment 1 - Dense",
        "Assignment 1 - Dense",
        "Assignment 1 - Dense",
    ],
    "Accuracy (%)": [78.60, 81.71, 82.88, 89.49, 89.49, 90.66, 91.44, 89.88],
})

In [ ]:
transformer_accuracy = results_df[["model", "accuracy"]].copy()
transformer_accuracy["Approach"] = "Assignment 2 - Transformer"
transformer_accuracy["Accuracy (%)"] = (transformer_accuracy["accuracy"] * 100).round(2)
transformer_accuracy = transformer_accuracy.rename(columns={"model": "Model"})[
    ["Model", "Approach", "Accuracy (%)"]
]

comparison_df = pd.concat([assignment1_results, transformer_accuracy], ignore_index=True)
comparison_df.sort_values("Accuracy (%)", ascending=False).reset_index(drop=True)


The transformer models all outperformed the best Assignment 1 model, which was Word2Vec + MLP with 91.44% accuracy.

RoBERTa achieved the best test accuracy and macro F1, slightly outperforming BERT. DistilBERT was weaker, but still clearly above the traditional NLP baselines from Assignment 1.

In [ ]:
plot_df = comparison_df.sort_values("Accuracy (%)", ascending=False).reset_index(drop=True)
baseline_accuracy = 91.44

palette = {
    "Assignment 1 - Sparse": "#4C78A8",
    "Assignment 1 - Dense": "#F58518",
    "Assignment 2 - Transformer": "#54A24B",
}

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(
    data=plot_df,
    x="Accuracy (%)",
    y="Model",
    hue="Approach",
    dodge=False,
    palette=palette,
    ax=ax,
)

ax.axvline(
    baseline_accuracy,
    color="#D62728",
    linestyle="--",
    linewidth=2,
    label="Best Assignment 1: 91.44%",
)
ax.text(
    baseline_accuracy + 0.25,
    len(plot_df) - 0.4,
    "Best Assignment 1\n91.44%",
    color="#D62728",
    fontsize=9,
    va="top",
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f%%", padding=4, fontsize=9)

ax.set_xlim(75, 100)
ax.set_title("Accuracy Comparison: Assignment 1 Baselines vs Transformers", pad=12)
ax.set_xlabel("Accuracy (%) - zoomed from 75% to 100%")
ax.set_ylabel("")
ax.grid(axis="x", linestyle=":", alpha=0.45)
ax.legend(title="", loc="lower center", bbox_to_anchor=(0.5, -0.22), ncol=4, frameon=False)
sns.despine(left=True, bottom=False)
plt.tight_layout()
plt.show()

# Error Analysis

In [ ]:
def analyze_model_errors(experiment_result, tokenized_dataset, num_examples=5):

    print(f"\n=== ERROR ANALYSIS: {experiment_result['model']} ===")

    predictions = experiment_result["predictions"]
    labels = experiment_result["labels"]


    error_indices = np.where(predictions != labels)[0]
    total_errors = len(error_indices)

    print(f"Total errors in the test set: {total_errors} out of {len(labels)} samples.")
    print(f"Sample of {num_examples} errors made by the model:\n")


    id2label_human = {
        0: "Ham (Legitimate)",
        1: "Phish (Attack)",
        2: "Spam"
    }


    sampled_errors = error_indices[:num_examples]

    for idx in sampled_errors:

        original_text = train_valid_test_dataset["test"][int(idx)]["text"]

        pred_label = id2label_human[predictions[idx]]
        real_label = id2label_human[labels[idx]]

        print(f"--- Error at Index {idx} ---")
        print(f"True Label:        {real_label}")
        print(f"Model Prediction:  {pred_label}")
        print(f"Email Text (Truncated):\n{original_text[:400]}...")
        print("-" * 40 + "\n")



if len(experiment_results) > 0:
    analyze_model_errors(experiment_results[0], train_valid_test_dataset)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


def plot_error_analysis(experiment_results):
    """Generates statistical plots focused on model error analysis."""

    # 1. PLOT: Total Error Count per Model
    models = [res["model"] for res in experiment_results]
    total_samples = len(experiment_results[0]["labels"])

    # Calculate the total number of errors for each model
    total_errors = [
        result["confusion_matrix"].sum()
        - np.diag(result["confusion_matrix"]).sum()
        for result in experiment_results
    ]

    plt.figure(figsize=(10, 5))
    sns.set_theme(style="whitegrid")

    barplot = sns.barplot(
        x=models,
        y=total_errors,
        palette="magma"
    )

    plt.title(
        f"Total Number of Classification Errors per Model "
        f"(Total Test Samples: {total_samples})"
    )
    plt.ylabel("Number of Misclassified Emails")
    plt.xlabel("Model / Configuration")

    # Add exact values on top of the bars
    for p in barplot.patches:
        barplot.annotate(
            format(p.get_height(), ".0f"),
            (p.get_x() + p.get_width() / 2.0, p.get_height()),
            ha="center",
            va="center",
            xytext=(0, 9),
            textcoords="offset points",
            fontweight="bold",
        )

    plt.tight_layout()
    plt.show()


    best_res = experiment_results[0]
    matrix = best_res["confusion_matrix"]


    error_matrix = matrix.copy()
    np.fill_diagonal(error_matrix, 0)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        error_matrix,
        annot=True,
        fmt="d",
        cmap="Reds",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
    )

    plt.title(f"Error Heatmap ({best_res['model']})")
    plt.ylabel("True Label (Actual Email Category)")
    plt.xlabel("Predicted Label (Model Prediction)")

    plt.text(
        0.5,
        -0.05,
        "Note: The diagonal was set to zero to visually emphasize classification errors only.",
        fontsize=10,
        style="italic",
        transform=plt.gca().transAxes,
        ha="center",
    )

    plt.tight_layout()
    plt.show()


# --- HOW TO RUN ---
# Pass the complete list of experiment results generated by your loop
plot_error_analysis(experiment_results)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


def analyze_model_errors(experiment_result, dataset_dict, num_examples=5):
    """Identifies and prints textual examples where the model failed."""
    print(f"\n=== TEXTUAL ERROR ANALYSIS: {experiment_result['model']} ===")

    predictions = experiment_result["predictions"]
    labels = experiment_result["labels"]


    error_indices = np.where(predictions != labels)[0]
    total_errors = len(error_indices)

    print(
        f"Total errors in the test set: "
        f"{total_errors} out of {len(labels)} samples."
    )
    print(f"Sample of {num_examples} errors made by the model:\n")

    # Mapping labels to more human-readable names
    id2label_human = {
        0: "Ham (Legitimate)",
        1: "Phish (Attack)",
        2: "Spam"
    }

    # Select the first error examples for analysis
    sampled_errors = error_indices[:num_examples]

    for idx in sampled_errors:
        # Retrieve the original email text from the test dataset
        original_text = dataset_dict["test"][int(idx)]["text"]

        pred_label = id2label_human[predictions[idx]]
        real_label = id2label_human[labels[idx]]

        print(f"--- Error at Index {idx} ---")
        print(f"True Label:        {real_label}")
        print(f"Model Prediction:  {pred_label}")
        print(f"Email Text (Truncated):\n{original_text[:400]}...")
        print("-" * 40 + "\n")


def plot_error_analysis(experiment_results):
    """Generates statistical plots focused on error analysis for ALL models."""

    models = [res["model"] for res in experiment_results]
    total_samples = len(experiment_results[0]["labels"])

    total_errors = [
        res["confusion_matrix"].sum()
        - np.diag(res["confusion_matrix"]).sum()
        for res in experiment_results
    ]

    plt.figure(figsize=(10, 5))
    sns.set_theme(style="whitegrid")

    barplot = sns.barplot(
        x=models,
        y=total_errors,
        palette="magma"
    )

    plt.title(
        f"Total Number of Errors per Model "
        f"(Total Test Samples: {total_samples})"
    )
    plt.ylabel("Number of Misclassified Emails")
    plt.xlabel("Model / Configuration")


    for p in barplot.patches:
        barplot.annotate(
            format(p.get_height(), ".0f"),
            (p.get_x() + p.get_width() / 2.0, p.get_height()),
            ha="center",
            va="center",
            xytext=(0, 9),
            textcoords="offset points",
            fontweight="bold",
        )

    plt.tight_layout()
    plt.show()


    for res in experiment_results:
        matrix = res["confusion_matrix"]

        # Create a copy of the matrix and zero the diagonal
        # (correct predictions) to highlight only errors
        error_matrix = matrix.copy()
        np.fill_diagonal(error_matrix, 0)

        plt.figure(figsize=(7, 5))

        sns.heatmap(
            error_matrix,
            annot=True,
            fmt="d",
            cmap="Reds",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
        )

        plt.title(f"Error Heatmap ({res['model']})")
        plt.ylabel("True Label (Actual Email Category)")
        plt.xlabel("Predicted Label (Model Prediction)")

        plt.text(
            0.5,
            -0.07,
            "Note: The diagonal was set to zero to emphasize only model errors.",
            fontsize=9,
            style="italic",
            transform=plt.gca().transAxes,
            ha="center",
        )

        plt.tight_layout()
        plt.show()


# =============================================================================
# RUNNING THE ANALYSIS
# =============================================================================


if len(experiment_results) > 0:
    analyze_model_errors(
        experiment_results[0],
        train_valid_test_dataset
    )


plot_error_analysis(experiment_results)